## This notebook is made to cover the practical part of the Machine Learning project that is related to creating a classification prediction model

Before introducing any kind of information in the dataset and/or features, the dataset **MUST BE SPLITTED**

Since we are dealing with a small unbalances dataset 4424 records, the split is going to be 80-20 with K-fold cross-validation to avoid problems with the target column since that is the feature we want to predict. The number of folds we do in the train data could also be changed to see which result we can get out of this parameter, first we are going to start with k = 10

In [13]:
import pandas as pd
from sklearn.model_selection import train_test_split

raw_dataframe = pd.read_csv('dropout_dataset.csv', sep=';')


#Before really splitting the data, some cleaning is done to avoid annoying errors in the future

for column in raw_dataframe.columns:
    raw_dataframe.rename(columns = {f'{column}':f'{column.rstrip().lstrip()}'})



X = raw_dataframe.drop(columns=["Target"])
y = raw_dataframe["Target"]



X_train, X_test, Y_train, Y_test = train_test_split(X,y,test_size=0.2, random_state=42)


Now that the split have been done, we need to define a strategy to deal with the imbalanced aspect of the dataset. In the reference paper, they used SMOTE, ADASYN and Logistic regression, regarding the avaliable resampling methods, we can try to apply 
- Oversampling techniques
  1. Borderline-SMOTE
  2. random over-sampling
  3. SMOTE-ENN
  4. SMOTENC (especifically designed for nominal and continuous features)

OR

- Algorithmic Solutions
  1. Weighted loss function
  2. Classical weight adjustment
  3. Focal loss


Since our dataset is small, there will be a very big loss of information if we do under-sampling because reducing even more the dataset will pose even more problems, the oversampling techniques will be used for this case

### Create a sample space using borderline SMOTE

In [17]:
from collections import Counter
from imblearn.over_sampling import BorderlineSMOTE

print('Original training dataset shape %s' % Counter(Y_train))

sm = BorderlineSMOTE(random_state=42)
X_borderline_SMOTE, Y_borderline_SMOTE = sm.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_borderline_SMOTE))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


### Create a sample space using random over sampler

In [18]:
from imblearn.over_sampling import RandomOverSampler

print('Original training dataset shape %s' % Counter(Y_train))

ros = RandomOverSampler(random_state=42)
X_random_over_sampler, Y_random_over_sampler = ros.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_random_over_sampler))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


### Create a sample space using random over SMOTE EEN

Here, the result is a little different than those we saw in the other resample stretegies

In [21]:
from imblearn.combine import SMOTEENN

print('Original training dataset shape %s' % Counter(Y_train))

sme = SMOTEENN(random_state=42)
X_smote_een, Y_smote_een = sme.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_smote_een))


Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Enrolled': 1154, 'Dropout': 868, 'Graduate': 613})


### Create a sample space using random over SMOTENC

Here, the result is a little different than those we saw in the other resample stretegies

In [26]:
from imblearn.over_sampling import SMOTENC

print('Original training dataset shape %s' % Counter(Y_train))

smnc = SMOTENC(random_state=42, categorical_features= [1,3,5,7])
X_smote_smnc, Y_smote_smnc = smnc.fit_resample(X_train,Y_train)

print('Resampled training dataset shape %s' % Counter(Y_smote_smnc))

Original training dataset shape Counter({'Graduate': 1791, 'Dropout': 1105, 'Enrolled': 643})
Resampled training dataset shape Counter({'Dropout': 1791, 'Enrolled': 1791, 'Graduate': 1791})


Regarding the algorithms to build the models, since our task is to classificate or fit into classification the three categories of students based on the given existing features:
- Success
- Relative success
- Failure

And knowing that in the reference study they used **Logistic regression, SVM, decision tree, random forest, Gradient boosting, Xtreme gradient boosting, legit boost and cat boost**. Some other avaliable options for algorithms are:
- **Probabilistic & Linear Models**
    1. Naïve Bayes (NB)
- **Neural Networks**
    1. Multi-Layer Perceptron (MLP - Feedforward Neural Network)
- **Rule-Based & Distance-Based Models**
    1. K-Nearest Neighbors (KNN)
- **Ensemble & Evolutionary Methods**
    1. Voting classifier
    2. Genetic algorithms

**1. First part of the study: For each model (or one selected model), we can test different sample spaces -> 4 x N_models** 
This will give the output of which sampling method should be used in the study

**2. Second part of the study: for each model, which one can give more satisfactory results -> N_models**
This will give the output of which model is more appropriate for this task

**3. For the most appropriate model, what k-fold is ideal for getting better results(evaluate if it makes sense)**

**4. Compare the results with the study results and draw conclusions**